# 📧 Mailchimp Marketing API Explorer

Enhanced API exploration using the official Mailchimp Marketing Python SDK

This notebook demonstrates:
- ✅ SDK setup and configuration
- ✅ Campaign analytics and reporting
- ✅ Audience management and segmentation
- ✅ Advanced data analysis techniques
- ✅ Data export and visualization

**Requirements:** 
- Mailchimp API key and server prefix
- Python packages: mailchimp-marketing, pandas, plotly, numpy


## 🔧 Setup & Configuration

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import json
from typing import Dict, List, Any

# Import Mailchimp SDK
import mailchimp_marketing as MailchimpMarketing
from mailchimp_marketing.api_client import ApiClientError

print("✅ Libraries imported successfully")

In [ ]:
# Configure Mailchimp client
# Set your credentials here or use environment variables

API_KEY = os.getenv('MAILCHIMP_API_KEY', 'YOUR_API_KEY_HERE')
SERVER = os.getenv('MAILCHIMP_SERVER', 'us1')  # e.g., us1, us2, eu1

# Initialize client
client = MailchimpMarketing.Client()
client.set_config({
    "api_key": API_KEY,
    "server": SERVER
})

# Test connection
try:
    response = client.ping.get()
    print(f"✅ Connected to Mailchimp API: {response}")
except ApiClientError as e:
    print(f"❌ Connection failed: {e}")
    print("Please check your API key and server settings")

## 📊 Campaign Analytics Deep-Dive

In [ ]:
# Get recent campaigns
def get_campaigns(status='sent', count=50):
    """Retrieve campaigns with enhanced filtering"""
    try:
        campaigns = client.campaigns.list(
            status=status,
            count=count,
            sort_field='send_time',
            sort_dir='DESC'
        )
        return campaigns.get('campaigns', [])
    except ApiClientError as e:
        print(f"Error retrieving campaigns: {e}")
        return []

campaigns = get_campaigns()
print(f"📧 Retrieved {len(campaigns)} campaigns")

# Display sample campaign info
if campaigns:
    sample = campaigns[0]
    print(f"\n📌 Sample Campaign:")
    print(f"  ID: {sample['id']}")
    print(f"  Title: {sample.get('settings', {}).get('title', 'N/A')}")
    print(f"  Subject: {sample.get('settings', {}).get('subject_line', 'N/A')}")
    print(f"  Send Time: {sample.get('send_time', 'N/A')}")

In [ ]:
# Get detailed campaign reports
def get_campaign_reports(campaign_ids):
    """Get comprehensive reports for multiple campaigns"""
    reports_data = []
    
    for campaign_id in campaign_ids[:10]:  # Limit to first 10 for demo
        try:
            report = client.reports.get_campaign_report(campaign_id)
            
            # Extract key metrics
            campaign_data = {
                'campaign_id': campaign_id,
                'campaign_title': report.get('campaign_title', 'Unknown'),
                'emails_sent': report.get('emails_sent', 0),
                'opens_total': report.get('opens', {}).get('opens_total', 0),
                'unique_opens': report.get('opens', {}).get('unique_opens', 0),
                'open_rate': report.get('opens', {}).get('open_rate', 0) * 100,
                'clicks_total': report.get('clicks', {}).get('clicks_total', 0),
                'unique_clicks': report.get('clicks', {}).get('unique_clicks', 0),
                'click_rate': report.get('clicks', {}).get('click_rate', 0) * 100,
                'bounces': report.get('bounces', {}).get('hard_bounces', 0) + 
                          report.get('bounces', {}).get('soft_bounces', 0),
                'bounce_rate': report.get('bounces', {}).get('bounce_rate', 0) * 100,
                'unsubscribes': report.get('unsubscribed', {}).get('unsubscribes', 0),
                'unsubscribe_rate': report.get('unsubscribed', {}).get('unsubscribe_rate', 0) * 100,
                'revenue': report.get('ecommerce', {}).get('total_revenue', 0),
                'send_time': report.get('send_time', '')
            }
            
            reports_data.append(campaign_data)
            
        except ApiClientError as e:
            print(f"⚠️ Could not get report for campaign {campaign_id}: {e}")
            continue
    
    return pd.DataFrame(reports_data)

# Get reports for recent campaigns
campaign_ids = [c['id'] for c in campaigns]
campaigns_df = get_campaign_reports(campaign_ids)

print(f"📊 Retrieved detailed reports for {len(campaigns_df)} campaigns")
campaigns_df.head()

In [ ]:
# Campaign Performance Analysis
if not campaigns_df.empty:
    # Performance metrics visualization
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('📬 Open vs Click Rate', '📊 Email Volume', 
                       '💰 Revenue Distribution', '📈 Engagement Trends'),
        specs=[[{}, {}],
               [{}, {}]]
    )
    
    # Open vs Click Rate Scatter
    fig.add_trace(
        go.Scatter(
            x=campaigns_df['open_rate'],
            y=campaigns_df['click_rate'],
            mode='markers',
            marker=dict(size=10, opacity=0.7),
            text=campaigns_df['campaign_title'],
            name='Campaigns'
        ),
        row=1, col=1
    )
    
    # Email Volume Bar Chart
    fig.add_trace(
        go.Bar(
            x=campaigns_df.index,
            y=campaigns_df['emails_sent'],
            name='Emails Sent'
        ),
        row=1, col=2
    )
    
    # Revenue Distribution
    fig.add_trace(
        go.Histogram(
            x=campaigns_df['revenue'],
            name='Revenue'
        ),
        row=2, col=1
    )
    
    # Engagement Trends (if send_time available)
    if 'send_time' in campaigns_df.columns:
        campaigns_df['send_date'] = pd.to_datetime(campaigns_df['send_time']).dt.date
        daily_metrics = campaigns_df.groupby('send_date').agg({
            'open_rate': 'mean',
            'click_rate': 'mean'
        }).reset_index()
        
        fig.add_trace(
            go.Scatter(
                x=daily_metrics['send_date'],
                y=daily_metrics['open_rate'],
                name='Avg Open Rate'
            ),
            row=2, col=2
        )
    
    fig.update_layout(height=600, showlegend=False, title_text="📊 Campaign Performance Dashboard")
    fig.show()
    
    # Performance summary
    print("\n📈 Performance Summary:")
    print(f"Average Open Rate: {campaigns_df['open_rate'].mean():.1f}%")
    print(f"Average Click Rate: {campaigns_df['click_rate'].mean():.1f}%")
    print(f"Total Revenue: ${campaigns_df['revenue'].sum():,.2f}")
    print(f"Total Emails Sent: {campaigns_df['emails_sent'].sum():,}")
else:
    print("No campaign data available for analysis")

## 👥 Audience Analytics

In [ ]:
# Get audience lists
def get_lists():
    """Retrieve all audience lists with stats"""
    try:
        lists_response = client.lists.get_all_lists(count=50)
        return lists_response.get('lists', [])
    except ApiClientError as e:
        print(f"Error retrieving lists: {e}")
        return []

lists = get_lists()
print(f"📋 Retrieved {len(lists)} audience lists")

# Process list data
lists_data = []
for list_item in lists:
    stats = list_item.get('stats', {})
    lists_data.append({
        'list_id': list_item['id'],
        'list_name': list_item.get('name', 'Unknown'),
        'member_count': stats.get('member_count', 0),
        'unsubscribe_count': stats.get('unsubscribe_count', 0),
        'cleaned_count': stats.get('cleaned_count', 0),
        'open_rate': stats.get('open_rate', 0) * 100,
        'click_rate': stats.get('click_rate', 0) * 100,
        'date_created': list_item.get('date_created', ''),
        'avg_sub_rate': stats.get('avg_sub_rate', 0),
        'avg_unsub_rate': stats.get('avg_unsub_rate', 0)
    })

lists_df = pd.DataFrame(lists_data)
print(f"\n📊 Processed {len(lists_df)} lists for analysis")
lists_df.head()

In [ ]:
# Audience Analysis Visualizations
if not lists_df.empty:
    # List performance comparison
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('📊 List Sizes', '📈 Engagement Rates', 
                       '⚖️ Growth vs Churn', '🎯 Performance Matrix'),
    )
    
    # List sizes
    fig.add_trace(
        go.Bar(
            x=lists_df['list_name'],
            y=lists_df['member_count'],
            name='Members'
        ),
        row=1, col=1
    )
    
    # Engagement rates
    fig.add_trace(
        go.Scatter(
            x=lists_df['list_name'],
            y=lists_df['open_rate'],
            mode='markers',
            name='Open Rate',
            marker=dict(size=10)
        ),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Scatter(
            x=lists_df['list_name'],
            y=lists_df['click_rate'],
            mode='markers',
            name='Click Rate',
            marker=dict(size=10)
        ),
        row=1, col=2
    )
    
    # Growth vs Churn
    fig.add_trace(
        go.Scatter(
            x=lists_df['avg_sub_rate'],
            y=lists_df['avg_unsub_rate'],
            mode='markers',
            text=lists_df['list_name'],
            name='Lists',
            marker=dict(size=lists_df['member_count']/100, opacity=0.7)
        ),
        row=2, col=1
    )
    
    # Performance Matrix
    fig.add_trace(
        go.Scatter(
            x=lists_df['open_rate'],
            y=lists_df['click_rate'],
            mode='markers',
            text=lists_df['list_name'],
            name='Performance',
            marker=dict(size=lists_df['member_count']/100, opacity=0.7)
        ),
        row=2, col=2
    )
    
    fig.update_layout(height=800, showlegend=False, title_text="👥 Audience Analytics Dashboard")
    fig.show()
    
    # Audience summary
    print("\n👥 Audience Summary:")
    print(f"Total Subscribers: {lists_df['member_count'].sum():,}")
    print(f"Average Open Rate: {lists_df['open_rate'].mean():.1f}%")
    print(f"Average Click Rate: {lists_df['click_rate'].mean():.1f}%")
    print(f"Total Lists: {len(lists_df)}")
    print(f"Largest List: {lists_df.loc[lists_df['member_count'].idxmax(), 'list_name']}")
else:
    print("No audience data available for analysis")

## 🔍 Advanced Analytics Examples

In [ ]:
# Geographic Analysis Example
def analyze_campaign_geography(campaign_id):
    """Analyze geographic performance for a specific campaign"""
    try:
        locations = client.reports.get_locations_for_campaign(campaign_id)
        
        if locations.get('countries'):
            geo_data = []
            for country in locations['countries']:
                geo_data.append({
                    'country': country.get('country', 'Unknown'),
                    'opens': country.get('opens', 0),
                    'clicks': country.get('clicks', 0),
                    'open_rate': country.get('open_rate', 0) * 100
                })
            
            geo_df = pd.DataFrame(geo_data)
            
            # Create geographic visualization
            fig = px.choropleth(
                geo_df,
                locations='country',
                color='opens',
                hover_data=['clicks', 'open_rate'],
                title=f"🌍 Geographic Performance - Campaign {campaign_id}",
                color_continuous_scale='Blues',
                locationmode='country names'
            )
            fig.show()
            
            return geo_df
    
    except ApiClientError as e:
        print(f"Could not analyze geography for campaign {campaign_id}: {e}")
        return None

# Analyze geography for the first campaign (if available)
if campaigns and len(campaigns) > 0:
    sample_campaign_id = campaigns[0]['id']
    geo_analysis = analyze_campaign_geography(sample_campaign_id)
    
    if geo_analysis is not None and not geo_analysis.empty:
        print(f"\n🌍 Top 5 Countries by Opens:")
        top_countries = geo_analysis.nlargest(5, 'opens')[['country', 'opens', 'open_rate']]
        for _, row in top_countries.iterrows():
            print(f"  {row['country']}: {row['opens']} opens ({row['open_rate']:.1f}% rate)")

In [ ]:
# Click Heatmap Analysis Example
def analyze_click_heatmap(campaign_id):
    """Analyze click performance for links in a campaign"""
    try:
        click_details = client.reports.get_campaign_click_details(campaign_id)
        
        if click_details.get('urls_clicked'):
            click_data = []
            for url in click_details['urls_clicked']:
                click_data.append({
                    'url': url.get('url', 'Unknown')[:50] + '...' if len(url.get('url', '')) > 50 else url.get('url', 'Unknown'),
                    'total_clicks': url.get('total_clicks', 0),
                    'unique_clicks': url.get('unique_clicks', 0),
                    'click_percentage': url.get('click_percentage', 0) * 100
                })
            
            click_df = pd.DataFrame(click_data)
            
            # Create click performance visualization
            fig = px.bar(
                click_df.head(10),
                x='unique_clicks',
                y='url',
                orientation='h',
                title=f"🔗 Click Heatmap - Campaign {campaign_id}",
                labels={'unique_clicks': 'Unique Clicks', 'url': 'Link URL'}
            )
            fig.update_layout(yaxis={'categoryorder': 'total ascending'})
            fig.show()
            
            return click_df
    
    except ApiClientError as e:
        print(f"Could not analyze clicks for campaign {campaign_id}: {e}")
        return None

# Analyze clicks for the first campaign (if available)
if campaigns and len(campaigns) > 0:
    click_analysis = analyze_click_heatmap(sample_campaign_id)
    
    if click_analysis is not None and not click_analysis.empty:
        print(f"\n🔗 Top 5 Links by Clicks:")
        top_links = click_analysis.nlargest(5, 'unique_clicks')[['url', 'unique_clicks', 'click_percentage']]
        for _, row in top_links.iterrows():
            print(f"  {row['url']}: {row['unique_clicks']} clicks ({row['click_percentage']:.1f}%)")

## 💾 Data Export & Integration

In [ ]:
# Export data for dashboard integration
def export_data_for_dashboard():
    """Export processed data for dashboard integration"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Export campaign data
    if not campaigns_df.empty:
        campaigns_df.to_csv(f'mailchimp_campaigns_{timestamp}.csv', index=False)
        campaigns_df.to_json(f'mailchimp_campaigns_{timestamp}.json', orient='records', date_format='iso')
        print(f"✅ Exported {len(campaigns_df)} campaigns to CSV and JSON")
    
    # Export audience data
    if not lists_df.empty:
        lists_df.to_csv(f'mailchimp_lists_{timestamp}.csv', index=False)
        lists_df.to_json(f'mailchimp_lists_{timestamp}.json', orient='records', date_format='iso')
        print(f"✅ Exported {len(lists_df)} lists to CSV and JSON")
    
    # Create summary report
    summary = {
        'export_timestamp': timestamp,
        'campaigns_count': len(campaigns_df) if not campaigns_df.empty else 0,
        'lists_count': len(lists_df) if not lists_df.empty else 0,
        'total_subscribers': lists_df['member_count'].sum() if not lists_df.empty else 0,
        'total_emails_sent': campaigns_df['emails_sent'].sum() if not campaigns_df.empty else 0,
        'avg_open_rate': campaigns_df['open_rate'].mean() if not campaigns_df.empty else 0,
        'avg_click_rate': campaigns_df['click_rate'].mean() if not campaigns_df.empty else 0,
        'total_revenue': campaigns_df['revenue'].sum() if not campaigns_df.empty else 0
    }
    
    with open(f'mailchimp_summary_{timestamp}.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"✅ Created summary report: mailchimp_summary_{timestamp}.json")
    
    return summary

# Export all data
export_summary = export_data_for_dashboard()

print("\n📋 Export Summary:")
for key, value in export_summary.items():
    if isinstance(value, (int, float)):
        if 'rate' in key:
            print(f"  {key}: {value:.1f}%")
        elif 'revenue' in key:
            print(f"  {key}: ${value:,.2f}")
        else:
            print(f"  {key}: {value:,}")
    else:
        print(f"  {key}: {value}")

## 🔮 Predictive Analytics Examples

In [ ]:
# Simple engagement prediction based on historical data
def predict_engagement_trends():
    """Basic engagement trend prediction"""
    
    if campaigns_df.empty or len(campaigns_df) < 3:
        print("Insufficient data for trend prediction")
        return
    
    # Calculate rolling averages for trend analysis
    if 'send_time' in campaigns_df.columns:
        campaigns_df['send_date'] = pd.to_datetime(campaigns_df['send_time'])
        campaigns_df = campaigns_df.sort_values('send_date')
        
        # Calculate 3-campaign rolling averages
        campaigns_df['open_rate_trend'] = campaigns_df['open_rate'].rolling(window=3).mean()
        campaigns_df['click_rate_trend'] = campaigns_df['click_rate'].rolling(window=3).mean()
        
        # Simple linear prediction for next campaign
        recent_open_trend = campaigns_df['open_rate_trend'].tail(3).mean()
        recent_click_trend = campaigns_df['click_rate_trend'].tail(3).mean()
        
        print("🔮 Engagement Predictions for Next Campaign:")
        print(f"  Predicted Open Rate: {recent_open_trend:.1f}%")
        print(f"  Predicted Click Rate: {recent_click_trend:.1f}%")
        
        # Trend visualization
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=campaigns_df['send_date'],
            y=campaigns_df['open_rate'],
            mode='markers',
            name='Open Rate',
            opacity=0.6
        ))
        
        fig.add_trace(go.Scatter(
            x=campaigns_df['send_date'],
            y=campaigns_df['open_rate_trend'],
            mode='lines',
            name='Open Rate Trend',
            line=dict(color='blue', width=3)
        ))
        
        fig.add_trace(go.Scatter(
            x=campaigns_df['send_date'],
            y=campaigns_df['click_rate_trend'],
            mode='lines',
            name='Click Rate Trend',
            line=dict(color='red', width=3)
        ))
        
        fig.update_layout(
            title="🔮 Engagement Trend Analysis",
            xaxis_title="Date",
            yaxis_title="Rate (%)",
            hovermode='x unified'
        )
        
        fig.show()
    else:
        print("Send time data not available for trend analysis")

predict_engagement_trends()

## 🎯 Action Items & Recommendations

In [ ]:
# Generate actionable insights and recommendations
def generate_recommendations():
    """Generate actionable insights based on data analysis"""
    
    recommendations = []
    
    # Campaign performance recommendations
    if not campaigns_df.empty:
        avg_open_rate = campaigns_df['open_rate'].mean()
        avg_click_rate = campaigns_df['click_rate'].mean()
        
        if avg_open_rate < 21.3:  # Industry average
            recommendations.append({
                'category': '📬 Open Rate Improvement',
                'priority': 'High',
                'action': f'Open rate ({avg_open_rate:.1f}%) is below industry average (21.3%). Consider A/B testing subject lines and send times.'
            })
        
        if avg_click_rate < 2.6:  # Industry average
            recommendations.append({
                'category': '🖱️ Click Rate Enhancement',
                'priority': 'Medium',
                'action': f'Click rate ({avg_click_rate:.1f}%) could be improved. Test different CTAs, content personalization, and email design.'
            })
        
        # Revenue optimization
        total_revenue = campaigns_df['revenue'].sum()
        if total_revenue == 0:
            recommendations.append({
                'category': '💰 Revenue Tracking',
                'priority': 'High',
                'action': 'No revenue data detected. Ensure ecommerce tracking is properly configured to measure ROI.'
            })
    
    # Audience recommendations
    if not lists_df.empty:
        total_subscribers = lists_df['member_count'].sum()
        avg_growth = lists_df['avg_sub_rate'].mean()
        avg_churn = lists_df['avg_unsub_rate'].mean()
        
        if avg_churn > avg_growth:
            recommendations.append({
                'category': '📉 List Health',
                'priority': 'High',
                'action': f'Churn rate ({avg_churn:.2f}%) exceeds growth rate ({avg_growth:.2f}%). Implement re-engagement campaigns and list cleaning.'
            })
        
        # Segmentation opportunities
        if len(lists_df) > 1:
            performance_variance = lists_df['open_rate'].std()
            if performance_variance > 10:
                recommendations.append({
                    'category': '🎯 Segmentation',
                    'priority': 'Medium',
                    'action': f'High variance in list performance ({performance_variance:.1f}%). Consider advanced segmentation for better targeting.'
                })
    
    # General recommendations
    recommendations.extend([
        {
            'category': '📊 Analytics Enhancement',
            'priority': 'Low',
            'action': 'Implement automated reporting dashboard using this SDK for real-time insights.'
        },
        {
            'category': '🔮 Predictive Analytics',
            'priority': 'Low',
            'action': 'Set up engagement prediction models to proactively identify at-risk subscribers.'
        }
    ])
    
    return recommendations

# Generate and display recommendations
recommendations = generate_recommendations()

print("🎯 Action Items & Recommendations:\n")
for i, rec in enumerate(recommendations, 1):
    priority_emoji = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}[rec['priority']]
    print(f"{i}. {rec['category']} {priority_emoji} {rec['priority']}")
    print(f"   {rec['action']}\n")

print("\n✅ Analysis Complete! Use this data to enhance your marketing dashboard and drive better campaign performance.")

---

## 📚 Next Steps for Dashboard Integration

This notebook demonstrates the enhanced capabilities available through the official Mailchimp Marketing SDK. To integrate with your dashboard:

### 🔧 **Technical Integration:**
1. **Configuration**: Use the config management system from `config/mailchimp_config.py`
2. **Data Ingestion**: Implement the enhanced ingestion script `scripts/ingest_mailchimp.py`
3. **Dashboard Pages**: Deploy the enhanced dashboard pages with advanced analytics

### 📊 **Key Enhancements Available:**
- **30+ additional metrics** not available in basic API calls
- **Real-time data updates** vs batch processing
- **Advanced segmentation** and audience insights
- **Geographic and click heatmap analysis**
- **Predictive analytics** capabilities
- **Revenue attribution** tracking

### 🎯 **Implementation Priority:**
1. **Phase 1**: Replace existing API calls with SDK (completed in this example)
2. **Phase 2**: Enhance dashboard pages with advanced visualizations
3. **Phase 3**: Implement predictive analytics and automated insights
4. **Phase 4**: Add real-time monitoring and alerting

**📧 Ready to transform your email marketing analytics!**